In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from collections import OrderedDict
import numpy as np
import qiskit_metal as metal
import math
import importlib
import airbridge_route

from shapely.ops import unary_union
from shapely.geometry import box, JOIN_STYLE
from qiskit_metal import draw, Dict
from qiskit_metal import designs
from qiskit_metal.qlibrary.core import QComponent
from qiskit_metal import Dict
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal import draw
from qiskit_metal.qlibrary.core.base import QComponent


# custom classes
from candycane_purcell import CandyCanePurcell
from candycane_purcell_meander import CandyCanePurcellMeander
from coupling_claw import CouplingClaw
from transmon import myTransmon
from squid import Squid
from transmon_waveguide import GiantAtomWaveguide
from airbridge_qcomponent import route_airbridgesQ, single_AirbridgeQ
from airbridge_route import _fillet_centerline, route_airbridges

In [ ]:
design = designs.design_planar.DesignPlanar()
design.overwrite_enabled = True
design._chips.main.size.size_x = '5mm'
design._chips.main.size.size_y = '5mm' 
gui = metal.MetalGUI(design)

08:56AM 57s INFO [_start_renderers]: Renderer=hfss skipped: runtime dependency not installed (QAnsysRenderer requires pyEPR. Install with: pip install 'quantum-metal[ansys]').
08:56AM 57s INFO [_start_renderers]: Renderer=q3d skipped: runtime dependency not installed (QAnsysRenderer requires pyEPR. Install with: pip install 'quantum-metal[ansys]').
08:56AM 57s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).
08:56AM 58s INFO [_start_renderers]: Renderer=aedt_q3d skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').
08:56AM 58s INFO [_start_renderers]: Renderer=aedt_hfss skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').


12:11PM 52s WARNING [_qt_message_handler]: WARNING: monitorData: Unable to obtain handle for monitor '\\.\DISPLAY1', defaulting to 96 DPI. (No context available from Qt)


In [ ]:
%metal_print LAUNCHPADS

In [5]:
# create wirebond pads:
# need to change based on LL rules (page 38, section 4.4)
# wirebond pad location is determined by the tip.
def create_wirebond(name, orientation, pos_y, pos_x, pad_width='150um', pad_length='100um', taper_height='200um', trace_width="5um",trace_gap="3um", layer="5"):
    options = Dict(
        orientation=orientation,
        pos_y=f"{pos_y}mm",
        pos_x=f"{pos_x}mm",
        pad_width=pad_width,
        # LaunchpadWirebond calls the straight rectangular length pad_height.
        pad_height=pad_length,
        taper_height=taper_height,
        trace_width=trace_width,
        trace_gap=trace_gap,
        layer=layer

    )
    return LaunchpadWirebond(design, name, options=options)

In [6]:
#Page 38, Section 4.4. Launchpads must be 200 um away from chip edge
#Only the launchpads this chip uses; names and positions are the same as in
#the full 24-pad layout (pad_i_x on the top/bottom edges, pad_i_y on the sides).
#create_wirebond(name, orientation, pos_y, pos_x)

#READOUT LINE: 10um trace / 6um gap tip to match the readout line (a 5/3 tip
#leaves a 0.5um ground notch where they join, M1.S.1)
pad_4_y = create_wirebond("pad_4_y", 0, 2, -2.0979, trace_width="10um", trace_gap="6um")
pad_10_y = create_wirebond("pad_10_y", 180, 2, 2.0979, trace_width="10um", trace_gap="6um")

#CHARGE LINES:
pad_6_y = create_wirebond("pad_6_y", 0, 0.4, -2.0979)   # flux line (SQUID)
#pad_12_y = create_wirebond("pad_12_y", 180, 0.4, 2.0979)

#FLUX LINES:
pad_3_y = create_wirebond("pad_3_y", 0, 1.2, -2.0979)   # charge line (top pad)


pad_9_y = create_wirebond("pad_9_y", 180, 1.2, 2.0979, trace_width="10um", trace_gap="6um")   # charge line (top pad)

#WAVEGUIDE: on the left and right edges at the waveguide's height
#(y = -0.0817mm), so the feeds run straight in to the qubits and the lower half
#of the chip is free for the waveguide's serpentines; 10/6 tip like the readout
#pad_wg_in = create_wirebond("pad_wg_in", 0, -0.0817, -2.0979, trace_width="10um", trace_gap="6um")
#pad_wg_out = create_wirebond("pad_wg_out", 180, -0.0817, 2.0979, trace_width="10um", trace_gap="6um")

gui.rebuild()
gui.autoscale()

In [ ]:
%metal_print CHARGE LINE

In [7]:
# CHARGE LINES: from launchpads at y = 1.2mm (pad_3_y on the left to an open end
# beside qubit's TOP pad. The bottom pad's sides belong to the waveguide's jogs, and
# a line from below would have to cross the waveguide's feed. The routes come
# down at x = -0.5mm, clear of the Purcell filters' jogs; the open end sits at y = -0.4mm

# LEFT:
open_charge_a = OpenToGround(
    design=design,
    name="open_charge",
    options=Dict(
        pos_x="-0.3mm",
        pos_y="-.4mm",
        orientation="0",
        width="5um",
        gap="3um",
        termination_gap="6um",
        layer="5",
    )
)

charge_line_a = RouteAnchors(
    design=design,
    name="charge_line",
    options=Dict(
        pin_inputs=Dict(
            start_pin=Dict(component="pad_3_y", pin="tie"),
            end_pin=Dict(component="open_charge", pin="open"),
        ),
        anchors=OrderedDict({
            0: np.array([-0.37, 1.2]),
            1: np.array([-0.37, -0.4]),     # same y as open_charge, so the last run is straight
        }),
        advanced=Dict(avoid_collision=False),   # the waveguide's bounding box covers this route
        fillet="50um",
        trace_width="5um",
        trace_gap="3um",
        layer="5",
    )
)


gui.rebuild()

In [8]:
%metal_print FLUX LINE


In [9]:
# FLUX LINES, like Ding et al., PRX 13, 031035 (2023): plain CPWs from the
# upper side launchpads (pad_6_y on the left, pad_12_y on the right, y = 0.4mm)
# to the qubit's "flux" pin, on the gap's centre line (y = 0). From there the
# qubit draws the PRX-style end: the trace narrows to 3um, the gaps flare out
# to 41um, and the trace is shorted to the ground strip at the edge of the
# loop's bay, so the current runs along that edge past the loop. The routes
# turn down at x = +/-1.5mm; the waveguide passes 82um below the line's end.
# As in the paper, the static Phi0/2 bias would come from a global coil and
# these lines do fine tuning; the mutual inductance needs a simulation.

# LEFT: flux line from pad_6_y to a shorted end beside the SQUID's connection arm.
# The SQUID (squid_test) is created in the instantiation cell at the end, after Q1,
# so the route is built by this function, which that cell calls.
def add_squid_flux_line(squid):
    # Get the arm's vertical center
    arm_y = (
        squid.p.attach_y
        + squid.p.side_arm_move_y
    )

    # Get the arm's outer attachment point
    arm_x = squid.pins["squid_center"]["middle"][0]

    # Position the flux-line termination
    flux_line_end = ShortToGround(
        design=design,
        name="squid_flux_end",
        options=Dict(
            pos_x=f"{arm_x - 0.025}mm",
            pos_y=f"{arm_y}mm",
            orientation="0",
            width="5um",
            gap="3um",
            termination_gap="0um",
            layer="5",
        )
    )

    flux_line_a = RouteAnchors(
        design=design,
        name="flux_line_squid",
        options=Dict(
            pin_inputs=Dict(
                start_pin=Dict(
                    component="pad_6_y",
                    pin="tie"
                ),
                end_pin=Dict(
                    component="squid_flux_end",
                    pin="short"
                )
            ),

            anchors=OrderedDict({
                0: np.array([-1.9,  0.4]),
                1: np.array([-1.9,  1.0]),
                2: np.array([-0.55,  1.0]),
                3: np.array([-0.55, -0.45]),
                4: np.array([-0.115, -0.45])
            }),
            advanced=Dict(avoid_collision=False),   
            fillet="50um",
            trace_width="5um",
            trace_gap="3um",
            layer="5",
        )
    )
    return flux_line_end, flux_line_a



In [10]:
%metal_print READOUT LINE

In [11]:
# READOUT LINE

readout_line = RouteStraight(
    design=design,
    name="readout_line",
    options=Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component="pad_4_y",
                pin="tie"
            ),
            end_pin=Dict(
                component="pad_10_y",
                pin="tie"
            ),
        ),
        advanced=Dict(
            avoid_collision=False
        ),
        trace_width="10um",
        trace_gap="6um",
        fillet="50um",
        layer="5",
    )
)

gui.rebuild()
gui.autoscale()

In [12]:
%metal_print PURCELL FILTER AND READOUT RESONATOR - VERIFY LENGTHS

In [13]:
purcell_and_res1 = CandyCanePurcellMeander(
    design=design,
    name="purcell_and_resonator_1",
    options=Dict(
        pos_x="0.189mm",
        pos_y="2.005mm",
        orientation="1",
        fillet="70um",
        meander_line_width="10um",
        si_fill="4um",
        down_from_pin_start="1522.8um",  # filter down to y ~ 0.30mm
        up_from_turn="812.3um",          # filter back up (tuned in the next cell)
        up_from_pin_start="880um",       # resonator_connect at y ~ 1.0mm
        down_from_turn="1164.6um",       # resonator back down (tuned in the next cell)
        meander_jogs="0",
        filter_meander_jogs="0",
    )
)

gui.rebuild()
gui.autoscale()

In [14]:
%metal_print TRANSMON AND COUPLING WAVEGUIDE

In [24]:
# instantiation
design.template_options.pop(myTransmon._get_unique_class_name(), None)
design.template_options.pop(GiantAtomWaveguide._get_unique_class_name(), None)

q1 = myTransmon(design, 'Q1', options=dict(pos_x='0um', pos_y='-0.6mm'))
wg = GiantAtomWaveguide(design=design, name="waveguide")

gui.rebuild()
gui.autoscale()


=== ARM DEBUG ===
side_arm_length = 0.041 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [16]:
%metal_print SQUID

In [17]:
# REFRESH SQUID COMPONENT TEMPLATE

template_key = f"{Squid.__module__}.{Squid.__name__}"

if template_key in design.template_options:
    for key, value in Squid.default_options.items():
        design.template_options[template_key][key] = value

# Create the SQUID with the corrected class
squid_test = Squid(
    design,
    "squid_test",
    options=Dict(
        stem_x=f"{q1.p.pos_x}mm",
        stem_width=f"{q1.p.trace_width}mm",
        attach_y=f"{q1.p.pos_y + q1.p.squid_offset}mm",
        transmon_side="1"
    )
)
print("SQUID attachment:", squid_test.pins["squid_center"]["middle"])
print("Extension:", squid_test.extension_plate_center)

# Flux line to the SQUID (defined in the FLUX LINE section)
flux_line_end, flux_line_a = add_squid_flux_line(squid_test)
gui.rebuild()


=== ARM DEBUG ===
side_arm_length = 0.041 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>
SQUID attachment: [-0.05067 -0.45   ]
Extension: (np.float64(-0.07317), np.float64(-0.464))
=== ARM DEBUG ===
side_arm_length = 0.041 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [18]:
%metal_print READOUT COUPLING

In [19]:
# coupling claw for readout
claw = CouplingClaw(
    design=design,
    name="coupling_claw",
    options=Dict(
        ref_component="Q1",
        ref_pin="readout",

        coupling_gap="4um",
        overlap_depth="30um",

        prong_width="16um",
        bridge_width="12um",

        lead_width="10um",

        layer="5",
    )
)

# qubit to readout coupling
cross1_res1 = RouteStraight(
    design=design,
    name="cross1_res1",
    options=Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component="purcell_and_resonator_1",
                pin="resonator_connect"
            ),
            end_pin=Dict(
                component="coupling_claw",
                pin="connect"
            )
        ),
        trace_width="10um",
        trace_gap="4um",
        layer="5",
    )
)

gui.rebuild()

=== ARM DEBUG ===
side_arm_length = 0.041 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [20]:
%metal_print AIRBRIDGES

In [21]:
# AIRBRIDGES on every CPW: the waveguide, the charge and flux lines, the readout
# line, the resonator-to-claw link, and the Purcell filter + readout resonator. They
# tie together the ground on both sides of the CPW (suppresses slotline modes).
# Same settings as chip_flux_qubits: 70um pitch, 10um from the ends, one on every
# bend. The waveguide's _airbridge_keepout_points keep them away from the qubit and
# the coupled runs; 90um clears the 58um pocket by more than a bridge's reach (26um



# from its centre to the end of a foot, plus the 1.5um margin of RR.E.1).
# Any bridge whose feet would land on an etched gap (a neighbouring line, the
# Purcell's T pad, the filter/resonator coupling region...) slides along its line
# until both feet are on ground, or is removed.
from shapely.ops import unary_union
from shapely.geometry import LineString, Point
from airbridge_route import _arclength_frame, _sample_at


def not_ground_region():
    """Everything on layer 5 that is etched (all subtract polygons and paths)."""
    polys = design.qgeometry.tables["poly"]
    paths = design.qgeometry.tables["path"]
    etch = [g for g, layer, sub in zip(polys.geometry, polys.layer, polys["subtract"])
            if str(layer) == "5" and sub]
    etch += [g.buffer(w/2, cap_style=2) for g, w, layer, sub in
             zip(paths.geometry, paths.width, paths.layer, paths["subtract"])
             if str(layer) == "5" and sub]
    return unary_union(etch)


def bridge_on_ground(bridge, not_ground):
    """True if every landing pad (layer 30) sits on ground with the foundry's
    1.5um margin (RR.E.1)."""
    polys = design.qgeometry.tables["poly"]
    pads = polys[(polys.component == bridge.id) & (polys.layer.astype(str) == "30")].geometry
    return all(pad.buffer(0.0015).intersection(not_ground).area == 0 for pad in pads)


def move_bridges_onto_ground(points, fillet, bridges, step_um=1, max_um=90):
    """Slide bridges whose landing pads aren't on ground along the rounded
    centerline of points (a route's, or one of the Purcell component's paths)
    until they are; delete the ones that find no spot within max_um."""
    centerline = _fillet_centerline(points, fillet)
    frame = _arclength_frame(centerline)
    line = LineString(centerline)
    not_ground = not_ground_region()
    moved, removed = [], []
    for bridge in bridges:
        if bridge_on_ground(bridge, not_ground):
            continue
        s0 = line.project(Point(design.parse_value(bridge.options.pos_x),
                                design.parse_value(bridge.options.pos_y)))
        for k in range(1, int(max_um / step_um) + 1):
            for sign in (1, -1):
                x, y, ori = _sample_at(frame, s0 + sign * k * step_um * 1e-3)
                bridge.options.pos_x = f"{x}mm"
                bridge.options.pos_y = f"{y}mm"
                bridge.options.orientation = ori
                bridge.rebuild()
                if bridge_on_ground(bridge, not_ground):
                    break
            else:
                continue
            moved.append((bridge.name, round(sign * k * step_um, 1)))
            break
        else:
            removed.append(bridge.name)
            design.delete_component(bridge.name)
    return moved, removed


def add_airbridges(route, keepout="90um", reserved_points=None):
    if reserved_points is None:
        reserved_points = getattr(route, "_airbridge_keepout_points", None)  # waveguide
    bridges = route_airbridges(
        design, route, pitch="70um", min_spacing="10um", bridge_at_corners=True,
        reserved_points=reserved_points, reserved_radius=keepout)
    fillet = design.parse_value(route.options.get("fillet", "0um"))
    moved, removed = move_bridges_onto_ground(route.get_points(), fillet, bridges)
    print(f"{route.name}: {len(bridges) - len(removed)} bridges; moved: {moved}; removed: {removed}")
    return [b for b in bridges if b.name not in removed]


def add_purcell_airbridges(purcell):
    """Bridges on both of the CandyCanePurcell's paths. Its CPW is meander_line_width
    wide with si_fill gaps; the span gets the same 3um of setback as the routes'."""
    p = purcell.p
    # route_airbridgesQ only replaces numbered names, so clear its bridges here
    for n in [n for n in design.components if n.startswith(f"{purcell.name}_ab_")]:
        design.delete_component(n)
    crossover = p.meander_line_width + 2 * p.si_fill + 0.003
    res_bridges, filt_bridges = route_airbridgesQ(
        design, purcell, pitch="70um", min_spacing="10um", bridge_at_corners=True,
        crossover_length=crossover)   # in mm
    kept = []
    # the component's "purcell" path is the readout resonator, "resonator" the filter
    for label, points, bridges in (("readout resonator", purcell.purcell_points(), res_bridges),
                                   ("Purcell filter", purcell.resonator_points(), filt_bridges)):
        moved, removed = move_bridges_onto_ground(points, p.fillet, bridges)
        print(f"{purcell.name} {label}: {len(bridges) - len(removed)} bridges; "
              f"moved: {moved}; removed: {removed}")
        kept += [b for b in bridges if b.name not in removed]
    return kept


wg_bridges = add_airbridges(wg)
cl_bridges = route_airbridges(design, charge_line_a, pitch="70um", min_spacing="10um", bridge_at_corners=True)
fl_bridges = route_airbridges(design, flux_line_a, pitch="70um", min_spacing="10um", bridge_at_corners=True)
# readout line: keep the bridges off the Purcell filter's T pad, which hangs
# below the line (its large pad is ~207um wide)
ro_bridges = add_airbridges(readout_line, keepout="140um",
                            reserved_points=[(purcell_and_res1.p.pos_x, purcell_and_res1.p.pos_y)])
link_bridges = add_airbridges(cross1_res1)
purcell_bridges = add_purcell_airbridges(purcell_and_res1)

gui.rebuild()

waveguide: 1216 bridges; moved: []; removed: []
readout_line: 55 bridges; moved: []; removed: []
cross1_res1: 17 bridges; moved: []; removed: []
purcell_and_resonator_1 readout resonator: 29 bridges; moved: []; removed: []
purcell_and_resonator_1 Purcell filter: 33 bridges; moved: []; removed: []
=== ARM DEBUG ===
side_arm_length = 0.041 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [22]:
wg.coupling_spacings
# should be 3e8/(sqrt(6.38))/(8e9)*3/2 = 22.26958825
# how much precision should we aim for? .1mm is 1% of the wavelength.

[np.float64(22.269600000000015),
 np.float64(22.26960000000002),
 np.float64(22.269599999999926),
 np.float64(22.269599999999002)]

In [28]:
# EXPORT: write the GDS from code (same as the spin-chain runner).
# chip_flux_qubits.gds is Qiskit Metal's raw export (ground on 5/0, pads and
# traces on 5/10 and 5/11). prepare_spin_chain_gds rebuilds layer 5 as
# ground - etches + signal metal, checks the airbridge landings, and
# flattens it into chip_flux_qubits_flat.gds, which is the file to send.
# (normalize_gds on its own keeps only 5/0 and drops the pads and traces.)
import os
from prepare_spin_chain_gds import prepare_spin_chain_gds

gds = design.renderers.gds
no_cheese = gds.options["no_cheese"]["view_in_file"]["main"]
cheese = gds.options["cheese"]["view_in_file"]["main"]
no_cheese[20] = True   # junction layers: never cheese
no_cheese[60] = True
cheese[20] = False
cheese[60] = False

raw_gds_path = "ga_8ghz.gds"
flat_gds_path = "ga_8ghz_flat.gds"

gds.export_to_gds(raw_gds_path)

report = prepare_spin_chain_gds(
    raw_gds_path,
    flat_gds_path,
    wg.get_points(),
    turn_radius=wg.p.fillet,                          # 45um, not the 50um default
    reserved_points=wg._airbridge_keepout_points,     # same keepout add_airbridges used
    reserved_radius=design.parse_value("90um"),
    layer5_signal_elements=list(
        gds.chip_info["main"][5]["q_subtract_false"]
    ),
    layer5_subtract_elements=list(
        gds.chip_info["main"][5]["q_subtract_true"]
    ),
)
print(report)
print("saved:", os.path.abspath(raw_gds_path))
print("saved:", os.path.abspath(flat_gds_path))

09:34AM 45s WARNING [import_junction_gds_file]: Not able to find file:"../resources/Fake_Junctions.GDS".  Not used to replace junction. Checked directory:"c:\Users\lamiy\Downloads\txtbks\Senior Proj\GA_8GHZ_IC_DESIGN\resources".
09:34AM 45s WARNING [_check_either_cheese]: layer=5 is not in chip=main either in no_cheese_view_in_file or cheese_view_in_file from self.options.
09:34AM 45s WARNING [_check_either_cheese]: layer=30 is not in chip=main either in no_cheese_view_in_file or cheese_view_in_file from self.options.
09:34AM 45s WARNING [_check_either_cheese]: layer=31 is not in chip=main either in no_cheese_view_in_file or cheese_view_in_file from self.options.


{'input_bridges': 1447, 'removed_invalid_bridges': 0, 'relocated_waveguide_turn_bridges': [], 'output_bridges': 1447, 'rr_posts': 2894, 'waveguide_turns': 78, 'waveguide_uturns': 68}
saved: c:\Users\lamiy\Downloads\txtbks\Senior Proj\GA_8GHZ_IC_DESIGN\design\ga_8ghz.gds
saved: c:\Users\lamiy\Downloads\txtbks\Senior Proj\GA_8GHZ_IC_DESIGN\design\ga_8ghz_flat.gds
